In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import re

plt.style.use('default')

# ── Trajectory (OptiTrack) files ──────────────────────────────────────────────
TRAJECTORY_FILES = r"""
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc1_Down.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc1_Ortho_2_perfect.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc2_Ortho_4.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc2_UpRedo_run3.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc3_Ortho_2.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc3_Up.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc4_Ortho.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Loc4_Down.csv
"""

# ── Log files ─────────────────────────────────────────────────────────────────
LOG_FILES = {
    'Loc1': {
        'non_ortho': r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc1_Down1.csv",
        'ortho':     r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc1_Ortho2_perfect.csv",
    },
    'Loc2': {
        'non_ortho': r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc2_UpRedo3.csv",
        'ortho':     r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc2_Ortho4.csv",
    },
    'Loc3': {
        'non_ortho': r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc3_Up1.csv",
        'ortho':     r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc3_Ortho2.csv",
    },
    'Loc4': {
        'non_ortho': r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc4_Down1.csv",
        'ortho':     r"C:\Users\ltjth\Documents\Research\UKF_Data\Good Runs\ARR_NC_Loc4_Ortho1.csv",
    },
}

# ── Column names ──────────────────────────────────────────────────────────────
COL_TIME     = 'Time'      # microseconds
COL_WIND     = 'Wind'      # m/s (already magnitude)
COL_BOUT_AMP = 'BoutAmp'   # bout amplitude from EWMA detector

BOUT_THRESHOLD = 25        # only accepted bouts above this value are shown

# ── Shared y-axis limits — adjust to fit your full dataset ───────────────────
WIND_YLIM = (0, 0.12)   # m/s
BOUT_YLIM = (0, 150)    # amplitude units

# ── Constants ─────────────────────────────────────────────────────────────────
FAN_X_RAW = -1514.199707 / 1000
FAN_Y_RAW =  2919.908203 / 1000

LOC_PALETTES = {
    'Loc1': ['#08519c', '#2171b5'],
    'Loc2': ['#006d2c', '#2ca25f'],
    'Loc3': ['#a50f15', '#de2d26'],
    'Loc4': ['#54278f', '#756bb1'],
}
DEFAULT_PALETTE = ['#636363', '#969696']

WIND_COLOR = 'black'
BOUT_COLOR = '#d94f00'

FONT_TRAJ = 30
FONT_TS   = 26
TICK_TRAJ = 28
TICK_TS   = 22


# ── Helpers ───────────────────────────────────────────────────────────────────

def get_color(loc_key, run_type):
    palette = LOC_PALETTES.get(loc_key, DEFAULT_PALETTE)
    return palette[0] if run_type == 'non_ortho' else palette[1]


def find_cfmrl_position_cols(filepath):
    with open(filepath) as f:
        lines = f.readlines()
    type_line  = lines[2].strip().split(',')
    name_line  = lines[3].strip().split(',')
    attr_line  = lines[5].strip().split(',')
    coord_line = lines[6].strip().split(',')
    x_col = y_col = None
    for i, (t, n, a, c) in enumerate(zip(type_line, name_line, attr_line, coord_line)):
        if n.strip() == 'CFMRL' and t.strip() == 'Rigid Body' and a.strip() == 'Position':
            if c.strip() == 'X': x_col = i
            elif c.strip() == 'Y': y_col = i
    return x_col, y_col


def load_log(path):
    try:
        df = pd.read_csv(path, low_memory=False)
        df.columns = df.columns.str.strip()
        return df
    except Exception as e:
        print(f"  [warn] Could not load {path}: {e}")
        return None


def plot_trajectory(ax, traj_entry, loc_key, run_type):
    ax.set_facecolor('white')
    rect = plt.Rectangle((-1, -1.0), 2, 0.75,
                         linewidth=1.5, edgecolor='black',
                         facecolor='gray', alpha=0.25)
    ax.add_patch(rect)

    if traj_entry is not None:
        label, path = traj_entry
        color = get_color(loc_key, run_type)
        x_col, y_col = find_cfmrl_position_cols(path)
        df = pd.read_csv(path, skiprows=6, header=0, low_memory=False)
        x = pd.to_numeric(df.iloc[:, x_col], errors='coerce')
        y = pd.to_numeric(df.iloc[:, y_col], errors='coerce')
        mask = x.notna() & y.notna()
        x = x[mask].values - FAN_X_RAW
        y = y[mask].values - FAN_Y_RAW
        ax.plot(x, y, color=color, linewidth=6, zorder=4)
        ax.scatter(x[0],  y[0],  color=color, marker='o', s=140,
                   zorder=5, edgecolors='black', linewidths=0.8)
        ax.scatter(x[-1], y[-1], color=color, marker='s', s=140,
                   zorder=5, edgecolors='black', linewidths=0.8)

    ax.scatter([], [], color='gray', marker='o', s=60,
               edgecolors='black', linewidths=0.8, label='Start')
    ax.scatter([], [], color='gray', marker='s', s=120,
               edgecolors='black', linewidths=0.8, label='End')
    ax.scatter(0, 0, color='yellow', marker='s', s=100,
               zorder=6, edgecolors='black', linewidths=1.0, label='Fan')

    ax.set_xlim(-4, 4)
    ax.set_ylim(-8, 0)
    ax.set_xlabel('X (m)', fontsize=FONT_TRAJ)
    ax.set_ylabel('Y (m)', fontsize=FONT_TRAJ)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.5, color='gray')
    ax.tick_params(labelsize=TICK_TRAJ)


def plot_timeseries(ax_w, log_path, loc_key, run_type):
    """
    Left axis  (ax_w): wind magnitude — black line, shared y-limits
    Right axis (ax_b): accepted bout amplitudes (amp > BOUT_THRESHOLD)
                       shown as discrete vertical bars, shared y-limits
    """
    ax_b = ax_w.twinx()
    ax_w.set_facecolor('white')

    df = load_log(log_path)
    if df is None:
        return ax_b

    # Time in seconds, zero-referenced
    t = pd.to_numeric(df[COL_TIME], errors='coerce').values
    t = (t - t[0]) * 1e-6   # µs → s

    # ── Wind ─────────────────────────────────────────────────────────────────
    if COL_WIND in df.columns:
        wind = pd.to_numeric(df[COL_WIND], errors='coerce').values
        ax_w.plot(t, wind, color=WIND_COLOR, linewidth=1.6, alpha=0.9, zorder=3)
        ax_w.set_ylabel('Wind (m/s)', color=WIND_COLOR, fontsize=FONT_TS)
        ax_w.tick_params(axis='y', labelcolor=WIND_COLOR, labelsize=TICK_TS)
        ax_w.set_ylim(WIND_YLIM)
        ax_w.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    else:
        print(f"  [warn] Column '{COL_WIND}' not found in {log_path}")

    # ── Accepted bouts — discrete vertical bars ───────────────────────────────
    if COL_BOUT_AMP in df.columns:
        bout = pd.to_numeric(df[COL_BOUT_AMP], errors='coerce').values

        # Find contiguous accepted-bout regions (amp > threshold)
        accepted = bout > BOUT_THRESHOLD
        changes  = np.diff(accepted.astype(int), prepend=0, append=0)
        starts   = np.where(changes ==  1)[0]
        ends     = np.where(changes == -1)[0]

        for s, e in zip(starts, ends):
            peak_amp = np.max(bout[s:e])
            t_s  = t[s]
            t_e  = t[min(e, len(t) - 1)]
            width = max(t_e - t_s, 0.3)   # min 0.3 s so thin bouts are visible
            ax_b.bar(
                t_s + width / 2,
                peak_amp,
                width=width,
                color=BOUT_COLOR,
                alpha=0.75,
                zorder=2,
                align='center',
            )

        ax_b.set_ylabel(f'Accepted Bouts',
                        color=BOUT_COLOR, fontsize=FONT_TS, labelpad=18)
        ax_b.tick_params(axis='y', labelcolor=BOUT_COLOR, labelsize=TICK_TS)
        ax_b.set_ylim(BOUT_YLIM)
    else:
        print(f"  [warn] Column '{COL_BOUT_AMP}' not found in {log_path}")

    ax_w.set_xlabel('Time (s)', fontsize=FONT_TS)
    ax_w.tick_params(axis='x', labelsize=TICK_TS)
    ax_w.grid(True, alpha=0.35, color='gray', zorder=0)
    ax_w.set_xlim(t[0], t[-1])

    return ax_b


# ── Build trajectory grid ─────────────────────────────────────────────────────
paths = [p.strip() for p in TRAJECTORY_FILES.strip().splitlines() if p.strip()]
files = {Path(p).stem: p for p in paths}

traj_grid = {f'Loc{i}': {'non_ortho': None, 'ortho': None} for i in range(1, 5)}
for label, path in files.items():
    m = re.search(r'Loc(\d+)', label, re.IGNORECASE)
    if m:
        key = f'Loc{m.group(1)}'
        if re.search(r'Ortho', label, re.IGNORECASE):
            traj_grid[key]['ortho'] = (label, path)
        else:
            traj_grid[key]['non_ortho'] = (label, path)

# ── Figure ────────────────────────────────────────────────────────────────────
# Equal column widths. Trajectory axes carry set_aspect('equal') so they
# become square; we resize the timeseries axes to match in a post-draw pass.

fig, axes = plt.subplots(
    4, 4,
    figsize=(32, 26),
    facecolor='white',
    gridspec_kw={'width_ratios': [1, 1, 1, 1], 'wspace': 0.75, 'hspace': 0.15}
)

LOC_LABELS = {
    'Loc1': 'Location 1',
    'Loc2': 'Location 2',
    'Loc3': 'Location 3',
    'Loc4': 'Location 4',
}
run_types  = ['non_ortho', 'ortho']
run_labels = ['Downwind / Upwind', 'Crosswind (Ortho)']

twin_axes = {}

for row, loc_key in enumerate(['Loc1', 'Loc2', 'Loc3', 'Loc4']):
    for col_pair, run_type in enumerate(run_types):
        traj_col = col_pair
        ts_col   = col_pair + 2

        ax_traj = axes[row, traj_col]
        ax_wind = axes[row, ts_col]

        plot_trajectory(ax_traj, traj_grid[loc_key][run_type], loc_key, run_type)



        log_path = LOG_FILES[loc_key][run_type]
        ax_bout  = plot_timeseries(ax_wind, log_path, loc_key, run_type)
        twin_axes[(row, ts_col)] = ax_bout

# ── Post-layout: resize timeseries panels to match trajectory panel height ────
fig.canvas.draw()

for row in range(4):
    for col_pair in range(2):
        traj_col = col_pair
        ts_col   = col_pair + 2

        ax_traj = axes[row, traj_col]
        ax_wind = axes[row, ts_col]
        ax_bout = twin_axes[(row, ts_col)]

        traj_pos = ax_traj.get_position()   # Bbox in figure fraction
        ts_pos   = ax_wind.get_position()

        new_pos = [ts_pos.x0, traj_pos.y0, ts_pos.width, traj_pos.height]
        ax_wind.set_position(new_pos)
        ax_bout.set_position(new_pos)   # twin must match

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Polygon
import matplotlib.patches as mpatches

plt.style.use('default')

# ── Trajectory (OptiTrack) files ──────────────────────────────────────────────
TRAJECTORY_FILES = r"""
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\ARR_NC_Shifting_miss.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\ARR_NC_Shifting_5.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Shifting_2_worked_manual_end.csv
C:\Users\ltjth\Documents\Research\UKF_Data\Optitrack\OSL\Good Runs\ARR_NC_Shifting_3_worked_auto_far.csv
"""

LOG_FILES = [
    r"C:\Users\ltjth\Documents\Research\UKF_Data\ARR_NC_Shifting1_miss.csv",
    r"C:\Users\ltjth\Documents\Research\UKF_Data\ARR_NC_Shifting5.csv",
    r"C:\Users\ltjth\Documents\Research\UKF_Data\ARR_NC_Shifting2_worked_manualend.csv",
    r"C:\Users\ltjth\Documents\Research\UKF_Data\ARR_NC_Shifting3_worked_auto_far.csv",
]

COL_TIME     = 'Time'
COL_WIND     = 'Wind'
COL_BOUT_AMP = 'BoutAmp'
BOUT_THRESHOLD = 25

WIND_YLIM = (0, 0.12)
BOUT_YLIM = (0, 150)
RUN_COLORS = ['#636363', '#969696', '#08519c', '#006d2c']
WIND_COLOR = 'black'
BOUT_COLOR = '#d94f00'

FONT_TRAJ = 30
FONT_TS   = 26
TICK_TRAJ = 28
TICK_TS   = 22

# ── Fan / landing zone configuration ─────────────────────────────────────────
FAN_START_POS    = (0.0, 0.0)       # origin
FAN_START_ANGLE  = 270              # degrees — fan 1 blows along +Y (upward)

FAN_END_POS      = (1, -0.5)     # hardcoded second fan position
FAN_END_ANGLE    = 135.0           # degrees — fan 2 heading

# Landing zone dimensions (drawn in front of each fan)
ZONE_WIDTH = 2.0    # metres, perpendicular to fan
ZONE_DEPTH = 0.75   # metres, along fan heading


# ── Helpers ───────────────────────────────────────────────────────────────────

def find_rigid_body_position_cols(filepath, body_name):
    with open(filepath) as f:
        lines = f.readlines()
    name_line  = lines[3].strip().split(',')
    attr_line  = lines[5].strip().split(',')
    coord_line = lines[6].strip().split(',')
    x_col = y_col = None
    for i, (n, a, c) in enumerate(zip(name_line, attr_line, coord_line)):
        if n.strip() == body_name and a.strip() == 'Position':
            if c.strip() == 'X': x_col = i
            elif c.strip() == 'Y': y_col = i
    return x_col, y_col


def load_optitrack(filepath):
    return pd.read_csv(filepath, skiprows=6, header=0, low_memory=False)


def get_optitrack_origin(filepath):
    """Return last valid Fan position as the world origin for this file."""
    x_col, y_col = find_rigid_body_position_cols(filepath, 'Fan')
    df = load_optitrack(filepath)
    fx = pd.to_numeric(df.iloc[:, x_col], errors='coerce').dropna()
    fy = pd.to_numeric(df.iloc[:, y_col], errors='coerce').dropna()
    return fx.iloc[-1], fy.iloc[-1]


def rotate(points, angle_deg):
    """Rotate Nx2 array of points by angle_deg around origin."""
    a = np.radians(angle_deg)
    c, s = np.cos(a), np.sin(a)
    R = np.array([[c, -s], [s, c]])
    return points @ R.T


def make_landing_zone(fan_pos, angle_deg, width=ZONE_WIDTH, depth=ZONE_DEPTH):
    """
    Build a landing zone rectangle placed in front of the fan.
    'In front' = along the fan's heading direction.
    Returns Nx2 array of world-frame vertices.
    """
    half_w = width / 2
    # Local frame: fan at origin, zone extends in +Y direction
    local = np.array([
        [-half_w, 0],
        [ half_w, 0],
        [ half_w, depth],
        [-half_w, depth],
    ])
    world = rotate(local, angle_deg) + np.array(fan_pos)
    return world


def load_log(path):
    try:
        df = pd.read_csv(path, low_memory=False)
        df.columns = df.columns.str.strip()
        return df
    except Exception as e:
        print(f"  [warn] Could not load {path}: {e}")
        return None


def get_fan1_pos(filepath):
    """Return first valid Fan position from this file, relative to the shared origin."""
    x_col, y_col = find_rigid_body_position_cols(filepath, 'Fan')
    df = load_optitrack(filepath)
    fx = pd.to_numeric(df.iloc[:, x_col], errors='coerce').dropna()
    fy = pd.to_numeric(df.iloc[:, y_col], errors='coerce').dropna()
    return fx.iloc[0], fy.iloc[0]


def plot_trajectory(ax, filepath, color, origin):
    ax.set_facecolor('white')

    origin_x, origin_y = origin

    # ── Fan 1 position from first datapoint of this file ─────────────────────
    fan1_raw_x, fan1_raw_y = get_fan1_pos(filepath)
    fan1_pos = (fan1_raw_x - origin_x, fan1_raw_y - origin_y)

    # ── Landing zone at fan 2 only ────────────────────────────────────────────
    zone2 = make_landing_zone(FAN_END_POS, FAN_END_ANGLE)
    ax.add_patch(Polygon(zone2, closed=True, linewidth=1.5,
                         edgecolor='black', facecolor='gray', alpha=0.25, zorder=2))

    # ── Fan markers ───────────────────────────────────────────────────────────
    # Fan 1 as rotated rectangle at its actual first position
    fan_size = 0.3
    fan1_local = np.array([
        [-fan_size/2, -fan_size/2],
        [ fan_size/2, -fan_size/2],
        [ fan_size/2,  fan_size/2],
        [-fan_size/2,  fan_size/2],
    ])
    fan1_world = rotate(fan1_local, FAN_START_ANGLE) + np.array(fan1_pos)
    ax.add_patch(Polygon(fan1_world, closed=True, linewidth=1.2,
                         edgecolor='black', facecolor='yellow', zorder=6))

    # Fan 2 as rotated rectangle at hardcoded position
    fan2_local = np.array([
        [-fan_size/2, -fan_size/2],
        [ fan_size/2, -fan_size/2],
        [ fan_size/2,  fan_size/2],
        [-fan_size/2,  fan_size/2],
    ])
    fan2_world = rotate(fan2_local, FAN_END_ANGLE) + np.array(FAN_END_POS)
    ax.add_patch(Polygon(fan2_world, closed=True, linewidth=1.2,
                         edgecolor='black', facecolor='orange', zorder=6))

    # Heading arrows on each fan
    for pos, angle, col in [(fan1_pos,    FAN_START_ANGLE, 'black'),
                             (FAN_END_POS, 225,             'darkorange')]:
        a = np.radians(angle)
        dx, dy = 0.5 * np.cos(a), 0.5 * np.sin(a)
        ax.annotate('', xy=(pos[0]+dx, pos[1]+dy), xytext=pos,
                    arrowprops=dict(arrowstyle='->', color=col, lw=2.0), zorder=8)

    # ── Drone trajectory ──────────────────────────────────────────────────────
    df = load_optitrack(filepath)
    x_col, y_col = find_rigid_body_position_cols(filepath, 'CFMRL')
    x = pd.to_numeric(df.iloc[:, x_col], errors='coerce')
    y = pd.to_numeric(df.iloc[:, y_col], errors='coerce')
    mask = x.notna() & y.notna()
    x = x[mask].values - origin_x
    y = y[mask].values - origin_y

    ax.plot(x, y, color=color, linewidth=6, zorder=4)
    ax.scatter(x[0],  y[0],  color=color, marker='o', s=140,
               zorder=5, edgecolors='black', linewidths=0.8)
    ax.scatter(x[-1], y[-1], color=color, marker='s', s=140,
               zorder=5, edgecolors='black', linewidths=0.8)

    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xlim(-6, 6)
    ax.set_ylim(-8, 0.5)
    ax.set_xlabel('X (m)', fontsize=FONT_TRAJ)
    ax.set_ylabel('Y (m)', fontsize=FONT_TRAJ)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(2))
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.5, color='gray')
    ax.tick_params(labelsize=TICK_TRAJ)

    # ── Legend ────────────────────────────────────────────────────────────────
    handles = [
        mpatches.Patch(facecolor='yellow', edgecolor='black', label='Fan 1'),
        mpatches.Patch(facecolor='orange', edgecolor='black', label='Fan 2'),
        plt.scatter([], [], color='gray', marker='o', s=60,
                    edgecolors='black', label='Drone start'),
        plt.scatter([], [], color='gray', marker='s', s=80,
                    edgecolors='black', label='Drone end'),
    ]
    ax.legend(handles=handles, fontsize=14, loc='upper left')


def plot_timeseries(ax_w, log_path):
    ax_b = ax_w.twinx()
    ax_w.set_facecolor('white')

    df = load_log(log_path)
    if df is None:
        return ax_b

    t = pd.to_numeric(df[COL_TIME], errors='coerce').values
    t = (t - t[0]) * 1e-6

    if COL_WIND in df.columns:
        wind = pd.to_numeric(df[COL_WIND], errors='coerce').values
        ax_w.plot(t, wind, color=WIND_COLOR, linewidth=1.6, alpha=0.9, zorder=3)
        ax_w.set_ylabel('Wind (m/s)', color=WIND_COLOR, fontsize=FONT_TS)
        ax_w.tick_params(axis='y', labelcolor=WIND_COLOR, labelsize=TICK_TS)
        ax_w.set_ylim(WIND_YLIM)
        ax_w.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

    if COL_BOUT_AMP in df.columns:
        bout = pd.to_numeric(df[COL_BOUT_AMP], errors='coerce').values
        accepted = bout > BOUT_THRESHOLD
        changes  = np.diff(accepted.astype(int), prepend=0, append=0)
        starts   = np.where(changes ==  1)[0]
        ends     = np.where(changes == -1)[0]
        for s, e in zip(starts, ends):
            peak_amp = np.max(bout[s:e])
            t_s  = t[s]
            t_e  = t[min(e, len(t) - 1)]
            width = max(t_e - t_s, 0.3)
            ax_b.bar(t_s + width / 2, peak_amp, width=width,
                     color=BOUT_COLOR, alpha=0.75, zorder=2, align='center')
        ax_b.set_ylabel(f'Accepted bout (amp > {BOUT_THRESHOLD})',
                        color=BOUT_COLOR, fontsize=FONT_TS, labelpad=18)
        ax_b.tick_params(axis='y', labelcolor=BOUT_COLOR, labelsize=TICK_TS)
        ax_b.set_ylim(BOUT_YLIM)

    ax_w.set_xlabel('Time (s)', fontsize=FONT_TS)
    ax_w.tick_params(axis='x', labelsize=TICK_TS)
    ax_w.grid(True, alpha=0.35, color='gray', zorder=0)
    ax_w.set_xlim(t[0], t[-1])

    return ax_b


# ── Main ──────────────────────────────────────────────────────────────────────
traj_paths = [p.strip() for p in TRAJECTORY_FILES.strip().splitlines() if p.strip()]
n_runs = len(traj_paths)

# Shared origin: last fan position from file 0
print("Reading origin from file 0...")
origin = get_optitrack_origin(traj_paths[0])
print(f"  Origin (raw): x={origin[0]:.3f}, y={origin[1]:.3f}")

fig, axes = plt.subplots(
    n_runs, 2,
    figsize=(18, 8 * n_runs),
    facecolor='white',
    gridspec_kw={'width_ratios': [1, 1], 'wspace': 0.75, 'hspace': 0.15}
)

twin_axes = {}

for row, (traj_path, log_path) in enumerate(zip(traj_paths, LOG_FILES)):
    ax_traj = axes[row, 0]
    ax_wind = axes[row, 1]
    color   = RUN_COLORS[row % len(RUN_COLORS)]

    plot_trajectory(ax_traj, traj_path, color, origin)
    ax_bout = plot_timeseries(ax_wind, log_path)
    twin_axes[row] = ax_bout

# ── Post-layout: match timeseries height to trajectory height ─────────────────
fig.canvas.draw()

for row in range(n_runs):
    ax_traj = axes[row, 0]
    ax_wind = axes[row, 1]
    ax_bout = twin_axes[row]

    traj_pos = ax_traj.get_position()
    ts_pos   = ax_wind.get_position()
    new_pos  = [ts_pos.x0, traj_pos.y0, ts_pos.width, traj_pos.height]
    ax_wind.set_position(new_pos)
    ax_bout.set_position(new_pos)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')

# ─── FILES & SOURCE PASS INDICES ─────────────────────────────────────────────
files = {
    '38F, Avg Windspeed 3.1 m/s':    (r'C:\Users\ltjth\Documents\Research\UKF_Data\ARR_outdoor_38F_alpha1_worked_passed_source.csv', 5600),
    '44F, Avg Windspeed 0.7 m/s': (r'C:\Users\ltjth\Documents\Research\UKF_Data\ARR_outdoor_44F4_initial approach worked_passed_source.csv', 4000),
}
# ─────────────────────────────────────────────────────────────────────────────

# Gather global axis limits across all runs
all_x, all_y, all_gas, all_bout = [], [], [], []
for path, _ in files.values():
    df = pd.read_csv(path)
    all_x.append(df['PosX'].values - df['PosX'].values[0])
    all_y.append(df['PosY'].values - df['PosY'].values[0])
    all_gas.append(df['Gas'].values)
    all_bout.append(df['BoutAmp'].values)

x_min = min(x.min() for x in all_x); x_max = max(x.max() for x in all_x)
y_min = min(y.min() for y in all_y); y_max = max(y.max() for y in all_y)
gas_min = min(g.min() for g in all_gas); gas_max = max(g.max() for g in all_gas)
bout_max = max(b.max() for b in all_bout)


# Square extent so both trajectory plots have identical x and y ranges
pad = 0.5
x_range = (x_max + pad) - (x_min - pad)
y_range = (y_max + pad) - (y_min - pad)
side = max(x_range, y_range)
x_mid = (x_min + x_max) / 2
y_mid = (y_min + y_max) / 2
x_lim = (x_mid - side / 2, x_mid + side / 2)
y_lim = (y_mid - side / 2, y_mid + side / 2)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor='white')
import matplotlib.lines as mlines

for row, (label, (path, split_idx)) in enumerate(files.items()):
    color = 'C0'
    df = pd.read_csv(path)
    x         = df['PosX'].values - df['PosX'].values[0]
    y         = df['PosY'].values - df['PosY'].values[0]
    t_sec     = (df['Time'].values - df['Time'].values[0]) / 1e6
    gas       = df['Gas'].values
    gas = pd.Series(gas).rolling(window=50, center=True, min_periods=1).mean().values
    bout_amp  = df['BoutAmp'].values
    flow_angle = df['FlowAngle'].values  # degrees, used as-is

    # ── Trajectory ───────────────────────────────────────────────────────────
    ax_traj = axes[row, 0]
    ax_traj.set_facecolor('white')

    ax_traj.plot(x[:split_idx+1], y[:split_idx+1], color=color, linewidth=0.8, alpha=0.85)
    ax_traj.plot(x[split_idx:],   y[split_idx:],   color=color, linewidth=0.8, alpha=0.85, linestyle='--')

    ax_traj.scatter(x[0],         y[0],         color=color, marker='o', s=60,  zorder=5, edgecolors='black', linewidths=0.8)
    ax_traj.scatter(x[split_idx], y[split_idx], color=color, marker='*', s=220, zorder=6, edgecolors='black', linewidths=0.8)
    ax_traj.scatter(x[-1],        y[-1],        color=color, marker='s', s=80,  zorder=6, edgecolors='black', linewidths=0.8)

    # Wind arrows
    arrow_len = 0.3
    for i in range(0, len(x), 200):
        angle_rad = np.deg2rad(flow_angle[i])+180%360
        dx = np.cos(angle_rad) * arrow_len
        dy = np.sin(angle_rad) * arrow_len
        ax_traj.annotate('',
            xy=(x[i] + dx, y[i] + dy), xytext=(x[i] - dx, y[i] - dy),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.0, mutation_scale=8))

    ax_traj.set_xlim(x_lim)
    ax_traj.set_ylim(y_lim)
    ax_traj.set_xlabel('X (m)', fontsize=11)
    ax_traj.set_ylabel('Y (m)', fontsize=11)
    ax_traj.set_title(label, fontsize=12)
    ax_traj.grid(True, alpha=0.3, linestyle='--', color='gray')


 # Trajectory legend — add wind_handle inside the list
    if row == 0:
        ax_traj.scatter([], [], color='gray', marker='o', s=60,  edgecolors='black', linewidths=0.8, label='Start')
        ax_traj.scatter([], [], color='gray', marker='*', s=200, edgecolors='black', linewidths=0.8, label='Passed Source')
        ax_traj.scatter([], [], color='gray', marker='s', s=80,  edgecolors='black', linewidths=0.8, label='End')
        ax_traj.plot([],   [],  color='gray', lw=0.8, ls='--', label='After Source')
        wind_handle = mlines.Line2D([], [], color='black', marker='>', markersize=6,
                                    linewidth=1.0, label='Wind Dir.')
        handles, labels = ax_traj.get_legend_handles_labels()
        ax_traj.legend(handles=handles + [wind_handle], fontsize=8, framealpha=0.9, loc='upper right')

    # ── Bouts & Gas ──────────────────────────────────────────────────────────
    ax_bout = axes[row, 1]
    ax_bout.set_facecolor('white')

    ax_bout = axes[row, 1]
    ax_bout.set_facecolor('white')

    bout_mask = bout_amp > 25
    ax_bout.vlines(t_sec[bout_mask], 0, bout_amp[bout_mask], color='C1', linewidth=1.2, alpha=0.9, label='Bout Amp (>25)')
    ax_bout.set_ylabel('Bout Amplitude', fontsize=11, color='C1')
    ax_bout.tick_params(axis='y', labelcolor='C1')
    ax_bout.set_ylim(0, bout_max * 1.15)   # ← single call, starts at -10
    ax_bout.set_xlim(0, t_sec[-1])
    ax_gas = ax_bout.twinx()                 # ← define BEFORE using it
    ax_gas.plot(t_sec, gas, color='gray', linewidth=0.6, alpha=0.3, label='Gas Conc.')
    ax_gas.set_ylabel('Gas Concentration', fontsize=11, color='gray')
    ax_gas.tick_params(axis='y', labelcolor='gray')
    ax_gas.set_ylim(-10, gas_max * 1.15)
    # ← aligned with bout axis

    # Source pass line
    ax_bout.axvline(t_sec[split_idx], color='black', linestyle='--', linewidth=1.2, label='Passed Source')

    ax_bout.set_xlabel('Time (s)', fontsize=11)
    ax_bout.set_title(f'{label} — Gas & Bouts', fontsize=12)
    ax_bout.grid(True, alpha=0.3, linestyle='--', color='gray')

    lines1, labels1 = ax_bout.get_legend_handles_labels()
    lines2, labels2 = ax_gas.get_legend_handles_labels()
    ax_bout.legend(lines1 + lines2, labels1 + labels2, fontsize=8, framealpha=0.9, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
####### MOTION COMPOSITE ########

import cv2
import numpy as np

def video_composite(video_path, output_path, every_n_frames=30, threshold=25):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    print(f"Video: {total_frames} frames, {fps:.1f} fps, {duration:.1f}s duration")

    start_sec = float(input(f"Start time in seconds (0 to {duration:.1f}): "))
    stop_sec = float(input(f"Stop time in seconds ({start_sec} to {duration:.1f}): "))

    start_frame = int(start_sec * fps)
    stop_frame = int(stop_sec * fps)

    # Seek to start frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    # Read frames in range
    all_frames = []
    for i in range(stop_frame - start_frame):
        ret, frame = cap.read()
        if not ret:
            break
        all_frames.append(frame)
    cap.release()
    print(f"Read {len(all_frames)} frames from {start_sec}s to {stop_sec}s")

    # Compute median background
    print("Computing background...")
    sample = np.array(all_frames[::5])
    background = np.median(sample, axis=0).astype(np.uint8)

    # Select frames to composite
    selected = all_frames[::every_n_frames]
    print(f"Compositing {len(selected)} frames...")

    # Build composite
    composite = background.copy()
    for i, frame in enumerate(selected):
        print(f"  Processing frame {i+1}/{len(selected)}")
        diff = cv2.absdiff(background, frame)
        gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
        mask = cv2.dilate(mask, np.ones((7, 7), np.uint8), iterations=2)
        composite[mask > 0] = frame[mask > 0]

    cv2.imwrite(output_path, composite)
    print(f"Saved to {output_path}")

video_composite(r"C:\Users\ltjth\Downloads\ICRA26_5490_VI_i.mp4", "icra26_timelapse2.png", every_n_frames=15)

In [ ]:
############# BOUT EXPLAINER ###################

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
import matplotlib as mpl

mpl.rcParams['font.family']      = 'sans-serif'
mpl.rcParams['font.sans-serif']  = ['DejaVu Sans']
mpl.rcParams['font.size']        = 20
mpl.rcParams['axes.titlesize']   = 9
mpl.rcParams['axes.labelsize']   = 15
mpl.rcParams['xtick.labelsize']  = 15
mpl.rcParams['ytick.labelsize']  = 15
mpl.rcParams['legend.fontsize']  = 12

# ── Colour palette ────────────────────────────────────────────────────────
C_SIGNAL  = '#2166AC'
C_BOUT    = '#D6604D'
C_DERIV1  = '#1A7A4A'
C_SHADE   = '#F4A582'
C_DERIV2  = '#542788'
C_VLINE_S = '#E08214'
C_THRESH  = '#8B0000'
C_ZERO    = '#888888'

# ── EWMA / bout-detection helper ──────────────────────────────────────────
def run_pipeline(gas, fs=100.0, hl=0.025, ampthresh=25, sd_thr=0.0):
    alpha = 1 - np.exp(-np.log(2) / (hl * fs))
    N = len(gas)
    xs      = np.zeros(N)
    xp_raw  = np.zeros(N)
    xp_s    = np.zeros(N)
    xpp_raw = np.zeros(N)
    xpp_s   = np.zeros(N)

    xs[0] = gas[0]
    for i in range(1, N):
        xs[i] = alpha * gas[i] + (1 - alpha) * xs[i-1]
    for i in range(1, N):
        xp_raw[i] = fs * (xs[i] - xs[i-1])
    xp_s[0] = xp_raw[0]
    for i in range(1, N):
        xp_s[i] = alpha * xp_raw[i] + (1 - alpha) * xp_s[i-1]
    for i in range(1, N):
        xpp_raw[i] = fs * (xp_s[i] - xp_s[i-1])
    xpp_s[0] = xpp_raw[0]
    for i in range(1, N):
        xpp_s[i] = alpha * xpp_raw[i] + (1 - alpha) * xpp_s[i-1]

    bout_regions = []
    in_bout, bout_start, bout_derivs = False, None, []
    for i in range(2, N):
        rising = xpp_s[i] > sd_thr
        if rising and not in_bout:
            in_bout, bout_start, bout_derivs = True, i, [xp_s[i]]
        elif rising and in_bout:
            bout_derivs.append(xp_s[i])
        elif not rising and in_bout:
            in_bout = False
            amp = max(bout_derivs) - min(bout_derivs)
            bout_regions.append({'start': bout_start, 'end': i,
                                  'amp': amp, 'accepted': amp > ampthresh})
            bout_derivs = []

    accepted = [b for b in bout_regions if b['accepted']]
    return xs, xp_s, xpp_s, accepted, alpha

# ── Parameters ────────────────────────────────────────────────────────────
fs        = 100.0
hl        = 0.025
ampthresh = 25
sd_thr    = 0.0

# ── Load primary (pipeline) file ──────────────────────────────────────────
PRIMARY = r"C:\Users\ltjth\Documents\Research\UKF_Data\Outside Data\outdoorBout_manualnotFlying_44F_avg0.41.csv"
df    = pd.read_csv(PRIMARY)
t_raw = (df['Time'] - df['Time'].iloc[0]) * 1e-6
gas   = df['Gas'].values.astype(float)
t     = t_raw.values
xs, xp_s, xpp_s, accepted_bouts, alpha = run_pipeline(gas, fs, hl, ampthresh, sd_thr)
print(f"Primary — accepted bouts: {len(accepted_bouts)}")

# ── Load right-column files ────────────────────────────────────────────────
RC_FILES = [
    r"C:\Users\ltjth\Documents\Research\UKF_Data\Outside Data\outdoorBout_manualnotFlying_44F_avg1.31.csv",
    r"C:\Users\ltjth\Documents\Research\UKF_Data\Outside Data\outdoorBout_manualnotFlying_44F_avg2.81.csv",
    r"C:\Users\ltjth\Documents\Research\UKF_Data\Outside Data\outdoorBout_manualnotFlying_44F_avg0.41.csv",
    r"C:\Users\ltjth\Documents\Research\UKF_Data\ARR_outdoor_38F_alpha1_worked_passed_source.csv",
]
RC_LABELS = ['44°F Avg Wind Speed: 1.3 ms$^{-1}$', '44°F Avg Wind Speed: 2.8 ms$^{-1}$', '44°F Avg Wind Speed: 0.4 ms$^{-1}$', '38°F Avg Wind Speed: 0.3 ms$^{-1}$']

rc_data = []
for path, lbl in zip(RC_FILES, RC_LABELS):
    d  = pd.read_csv(path)
    tr = (d['Time'] - d['Time'].iloc[0]) * 1e-6
    g  = d['Gas'].values.astype(float)
    xs_rc, _, _, bouts, _ = run_pipeline(g, fs, hl, ampthresh, sd_thr)
    rc_data.append({'t': tr.values, 'gas': g, 'xs': xs_rc, 'bouts': bouts, 'label': lbl})
    print(f"{lbl} — accepted bouts: {len(bouts)}")

# ── Y-axis labels ─────────────────────────────────────────────────────────
YLABELS_L = [
    'ADC counts',
    r"$x^\prime_s$ (ADC s$^{-1}$)",
    r"$x^{\prime\prime}_s$ (ADC s$^{-2}$)",
    r'Bout Amplitude (ADC s$^{-1}$)',
]

# ── Figure layout: 4 rows x 2 columns ────────────────────────────────────
fig = plt.figure(figsize=(14.4, 10.5))
fig.patch.set_facecolor('white')
gs = GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.32,
              top=0.93, bottom=0.06, left=0.07, right=0.97)

axes_l = [fig.add_subplot(gs[i, 0]) for i in range(4)]
axes_r = [fig.add_subplot(gs[i, 1]) for i in range(4)]

def style_ax(ax, ylabel, last_row):
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color('0.3')
    ax.tick_params(which='both', direction='in', top=True, right=True, pad=3)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(4))
    ax.grid(which='major', linestyle=':', linewidth=0.4, color='0.82', zorder=0)
    ax.set_ylabel(ylabel, labelpad=4)
    if not last_row:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel('Time (s)', labelpad=3)

# ══════════════════════════════════════════════════════════════════════════
# LEFT COLUMN — pipeline panels
# ══════════════════════════════════════════════════════════════════════════

# Panel 0: EWMA
ax = axes_l[0]
ax.plot(t, xs, color=C_SIGNAL, linewidth=1.1, zorder=3,
        label=rf'EWMA Smoothed Gas Concentration')
added = False
for b in accepted_bouts:
    seg = slice(b['start'], b['end'] + 1)
    lbl = 'Accepted bout' if not added else '_'
    added = True
    ax.plot(t[seg], gas[seg], color=C_BOUT, linewidth=1.8, zorder=4, label=lbl)
ax.legend(loc='upper left', handlelength=1.5, borderpad=0.5, frameon=True)
style_ax(ax, YLABELS_L[0], False)

# Panel 1: First derivative
ax = axes_l[1]
ax.axhline(0, color=C_ZERO, linewidth=0.7, linestyle='--', zorder=1)
ax.plot(t, xp_s, color=C_DERIV1, linewidth=0.8, alpha=0.95, zorder=3)
added = False
for b in accepted_bouts:
    lbl = 'Bout region' if not added else '_'
    added = True
    ax.axvspan(t[b['start']], t[b['end']],
               alpha=0.22, color=C_SHADE, zorder=2, label=lbl)
ax.legend(loc='upper left', handlelength=1.5, borderpad=0.5, frameon=True)
style_ax(ax, YLABELS_L[1], False)

# Panel 2: Second derivative
ax = axes_l[2]
ax.plot(t, xpp_s, color=C_DERIV2, linewidth=0.8, alpha=0.95, zorder=3)
added = False
for b in accepted_bouts:
    lbl = 'Bout region' if not added else '_'
    added = True
    ax.axvspan(t[b['start']], t[b['end']],
               alpha=0.22, color=C_SHADE, zorder=2, label=lbl)
ax.legend(loc='upper left', handlelength=1.5, borderpad=0.5, frameon=True)
style_ax(ax, YLABELS_L[2], False)

# Panel 3: Bout amplitudes
ax = axes_l[3]
if accepted_bouts:
    bout_t   = np.array([t[b['start']] for b in accepted_bouts])
    bout_amp = np.array([b['amp']      for b in accepted_bouts])
    markerline, stemlines, baseline = ax.stem(
        bout_t, bout_amp, markerfmt='o', basefmt=' ')
    plt.setp(stemlines, color=C_VLINE_S, linewidth=1.0, alpha=0.75)
    markerline.set_markerfacecolor(C_BOUT)
    markerline.set_markeredgecolor('white')
    markerline.set_markeredgewidth(0.5)
    markerline.set_markersize(5)
    ax.axhline(ampthresh, color=C_THRESH, linewidth=0.9, linestyle='--',
               label=rf'$b_{{thr}} = {ampthresh}$ ADC s$^{{-1}}$', zorder=3)
    ax.set_ylim(bottom=0)
    ax.legend(loc='upper left', handlelength=1.5, borderpad=0.5, frameon=True)
style_ax(ax, YLABELS_L[3], True)

# ══════════════════════════════════════════════════════════════════════════
# RIGHT COLUMN — raw gas + detected bouts for 4 files
# ══════════════════════════════════════════════════════════════════════════
for row, (ax, d) in enumerate(zip(axes_r, rc_data)):
    tr, g, xs_rc, bouts, lbl = d['t'], d['gas'], d['xs'], d['bouts'], d['label']
    ax.plot(tr, xs_rc, color=C_SIGNAL, linewidth=1.1, zorder=3)
    added = False
    for b in bouts:
        seg = slice(b['start'], b['end'] + 1)
        bl = 'Accepted bout' if not added else '_'
        added = True
        ax.plot(tr[seg], g[seg], color=C_BOUT, linewidth=1.6, zorder=4, label=bl)
    ax.legend(loc='upper left', handlelength=1.5, borderpad=0.5, frameon=True)
    style_ax(ax, 'ADC counts', row == 3)
    ax.set_title(lbl, fontsize=13, loc='left', pad=3, style='italic')

plt.show()

In [ ]:
"""
Wind Comparison Analysis - Multi-Speed Comparison
Compares UKF, Crazyflie velocity, and model estimates across different speeds and trajectories.
Layout: Line No Wind | Line With Wind | Square No Wind | Square With Wind
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from filterpy.kalman import UnscentedKalmanFilter as UKF
from filterpy.kalman import MerweScaledSigmaPoints
from scipy.signal import butter, filtfilt


# =============================================================================
# Data Loading
# =============================================================================

def lowpass_filter(data, cutoff=5.0, fs=100.0, order=4):
    """Apply Butterworth low-pass filter."""
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)


def load_data(filepath):
    """Load flight data CSV and preprocess magnetic field measurements."""
    df = pd.read_csv(filepath)

    # Filter magnetic field data
    for axis in ['Bx', 'By']:
        df[f'{axis}_filt'] = lowpass_filter(df[axis].values)

    # Use filtered values
    df['Bx_cf'] = df['Bx_filt']
    df['By_cf'] = df['By_filt']

    # Compute magnitudes
    df['B_mag'] = np.sqrt(df['Bx_cf'] ** 2 + df['By_cf'] ** 2)
    df['Bxy_mag'] = np.sqrt(df['Bx_cf'] ** 2 + df['By_cf'] ** 2)

    # Compute qw if quaternion components exist
    if all(col in df.columns for col in ['Qx', 'Qy', 'Qz']):
        qw_sq = 1.0 - (df['Qx'] ** 2 + df['Qy'] ** 2 + df['Qz'] ** 2)
        df['qw'] = np.sqrt(np.maximum(qw_sq, 0))

    # Compute velocity magnitude
    if 'Vx' in df.columns:
        df['V_mag'] = np.sqrt(df['Vx'] ** 2 + df['Vy'] ** 2)

    return df


# =============================================================================
# Sensor Model
# =============================================================================

class QuadraticSensorModel:
    """Quadratic sensor model mapping airflow velocity to magnetic field."""

    def __init__(self):
        # Inverse model coefficients: B_mag -> v_mag
        self.inv_a = -0.000213
        self.inv_b = 0.03955
        self.inv_c = 0.03621

        # Forward model coefficients: v_mag -> B_mag
        self.fwd_a = 29.47
        self.fwd_b = -7.7
        self.fwd_c = 5.44

    def velocity_to_B(self, v_mag):
        """Predict B magnitude from airflow speed."""
        B = self.fwd_a * v_mag ** 2 + self.fwd_b * v_mag + self.fwd_c
        return np.maximum(B, 0)

    def B_to_velocity(self, B_mag):
        """Estimate airflow speed from B magnitude."""
        v = self.inv_a * B_mag ** 2 + self.inv_b * B_mag + self.inv_c
        return np.maximum(v, 0)


# =============================================================================
# Airflow Estimation UKF
# =============================================================================

class AirflowUKF:
    """UKF estimating relative airflow velocity in body frame."""

    def __init__(self, dt, sensor_model, Q_airflow=0.5, R_magnetic=None, R_odom=0.003):
        self.dt = dt
        self.sensor_model = sensor_model
        self.R_odom = R_odom

        self.dim_x = 2  # [va_x, va_y]
        self.dim_z = 2  # [Bx, By]

        points = MerweScaledSigmaPoints(n=self.dim_x, alpha=0.1, beta=2.0, kappa=0.0)

        self.ukf = UKF(
            dim_x=self.dim_x,
            dim_z=self.dim_z,
            dt=dt,
            fx=self._fx,
            hx=self._hx,
            points=points
        )

        self.ukf.x = np.zeros(2)
        self.ukf.P = np.eye(2) * 1.0
        self.ukf.Q = np.eye(2) * Q_airflow

        if R_magnetic is not None:
            self.ukf.R = R_magnetic
        else:
            self.ukf.R = np.array([
                [0.72422324, 0.33065],
                [0.33065, 0.76509166]
            ])

        self._accel_body = np.zeros(2)

    def _fx(self, x, dt):
        return x  # Random walk

    def _hx(self, x):
        va_x, va_y = x
        v_mag = np.sqrt(va_x ** 2 + va_y ** 2)

        c = 29.47
        b = -7.7

        Bx = c * v_mag * va_x + b * va_x
        By = c * v_mag * va_y + b * va_y

        return np.array([Bx, By])

    def predict(self, accel_body_xy=None):
        if accel_body_xy is not None:
            self._accel_body = np.asarray(accel_body_xy)

        self.ukf.P = 0.5 * (self.ukf.P + self.ukf.P.T)
        self.ukf.P += np.eye(self.dim_x) * 1e-6
        self.ukf.predict()

    def update_with_velocity(self, vx_body, vy_body):
        """Soft constraint from odometry."""
        z = np.array([-vx_body, -vy_body])
        H = np.eye(2)
        R_odom = np.eye(2) * self.R_odom

        y = z - self.ukf.x
        S = H @ self.ukf.P @ H.T + R_odom
        K = self.ukf.P @ H.T @ np.linalg.inv(S)

        self.ukf.x = self.ukf.x + K @ y
        self.ukf.P = (np.eye(2) - K @ H) @ self.ukf.P

    def update(self, Bx, By):
        z = np.array([Bx, By])

        self.ukf.P = 0.5 * (self.ukf.P + self.ukf.P.T)
        self.ukf.P += np.eye(self.dim_x) * 1e-6

        try:
            self.ukf.update(z)
        except np.linalg.LinAlgError:
            self.ukf.P = np.eye(self.dim_x) * 1.0

    @property
    def airflow(self):
        return self.ukf.x.copy()


# =============================================================================
# Run UKF
# =============================================================================

def run_ukf(df, Q_airflow=0.005, R_magnetic=None, R_odom=0.003):
    """Run UKF on a dataset and return results."""

    # Determine timestep
    if 'time' in df.columns:
        time = df['time'].values
        if time[0] > 1000:
            time = time / 1000.0
        dt = np.median(np.diff(time))
    else:
        dt = 0.02
        time = np.arange(len(df)) * dt

    time = time - time[0]

    # Initialize
    sensor_model = QuadraticSensorModel()
    ukf = AirflowUKF(dt, sensor_model, Q_airflow=Q_airflow,
                     R_magnetic=R_magnetic, R_odom=R_odom)

    N = len(df)
    results = {
        'time': time,
        'va_x': np.zeros(N),
        'va_y': np.zeros(N),
        'va_mag': np.zeros(N),
        'v_empirical': np.zeros(N),
        'vx_true': df['Vx'].values,
        'vy_true': df['Vy'].values,
        'v_true_mag': np.sqrt(df['Vx'] ** 2 + df['Vy'] ** 2).values,
        'Bx': df['Bx_cf'].values,
        'By': df['By_cf'].values,
        'B_mag': df['B_mag'].values,
        'wind_x': np.zeros(N),
        'wind_y': np.zeros(N),
        'wind_mag': np.zeros(N),
    }

    # Run filter
    for i in range(N):
        ukf.predict()

        Bx = df['Bx_cf'].iloc[i]
        By = df['By_cf'].iloc[i]
        ukf.update(Bx, By)
        ukf.update_with_velocity(df['Vx'].iloc[i], df['Vy'].iloc[i])

        va = ukf.airflow
        results['va_x'][i] = va[0]
        results['va_y'][i] = va[1]
        results['va_mag'][i] = np.sqrt(va[0] ** 2 + va[1] ** 2)

        results['v_empirical'][i] = sensor_model.B_to_velocity(df['B_mag'].iloc[i])

        # Wind = airflow + velocity
        results['wind_x'][i] = va[0] + df['Vx'].iloc[i]
        results['wind_y'][i] = va[1] + df['Vy'].iloc[i]
        results['wind_mag'][i] = np.sqrt(results['wind_x'][i] ** 2 + results['wind_y'][i] ** 2)

    return results


# =============================================================================
# Plotting
# =============================================================================

def plot_side_by_side_comparison(results_no_wind, results_wind, speeds,
                                 xlim_line_no_wind, xlim_square_no_wind,
                                 xlim_line_wind, xlim_square_wind, fontsize=11):
    """
    Create comparison with columns: Line No Wind | Line With Wind | Square No Wind | Square With Wind

    Parameters
    ----------
    results_no_wind : dict
        Results dictionary for no-wind conditions
    results_wind : dict
        Results dictionary for with-wind conditions
    speeds : list
        List of speeds
    xlim_line_no_wind, xlim_square_no_wind : dict
        X-axis limits for no-wind conditions
    xlim_line_wind, xlim_square_wind : dict
        X-axis limits for with-wind conditions
    fontsize : int
        Font size for labels
    """

    n_speeds = len(speeds)
    fig = plt.figure(figsize=(18, 2.5 * n_speeds))

    gs = fig.add_gridspec(n_speeds, 4, hspace=0.3, wspace=0.3)

    ylim = 2.5

    # Colors
    color_odom = '#000000'
    color_ukf = '#0d42db'
    color_model = '#7f7f7f'

    # Define column layout: [trajectory, condition, xlim_dict]
    columns = [
        ('Line', 'no_wind', xlim_line_no_wind, 'Line - No Wind'),
        ('Line', 'wind', xlim_line_wind, 'Line - With Wind'),
        ('Square', 'no_wind', xlim_square_no_wind, 'Square - No Wind'),
        ('Square', 'wind', xlim_square_wind, 'Square - With Wind')
    ]

    for col_idx, (traj, condition, xlim_dict, title) in enumerate(columns):
        for row, speed in enumerate(speeds):
            ax = fig.add_subplot(gs[row, col_idx])

            # Select the appropriate results dictionary
            results_dict = results_no_wind if condition == 'no_wind' else results_wind

            # Check if data exists
            if speed not in results_dict or traj not in results_dict[speed]:
                ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                       transform=ax.transAxes, fontsize=fontsize)
                ax.set_xlim([0, xlim_dict.get(speed, 17)])
                ax.set_ylim([0, ylim])
            else:
                results = results_dict[speed][traj]

                # Plot
                ax.plot(results['time'], results['v_true_mag'],
                       color=color_odom, lw=2, label='Crazyflie', alpha=0.8)
                ax.plot(results['time'], results['va_mag'],
                       color=color_ukf, lw=2, label='UKF', alpha=0.8)
                ax.plot(results['time'], results['v_empirical'],
                       color=color_model, lw=1.5, label='Model', alpha=0.6, linestyle='--')

                ax.set_xlim([0, xlim_dict.get(speed, 17)])
                ax.set_ylim([0, ylim])
                ax.grid(True, alpha=0.3)

            # Labels
            if row == n_speeds - 1:
                ax.set_xlabel('Time (s)', fontsize=fontsize)
            if row == 0:
                ax.set_title(title, fontsize=fontsize + 1, fontweight='bold')
            # Y-label only on leftmost column
            if col_idx == 0:
                ax.set_ylabel(f'{speed} m/s\nSpeed (m/s)', fontsize=fontsize)

                # Add this right before 'return fig' in the plot_side_by_side_comparison function

                # Create legend at the top of the figure
                handles, labels = ax.get_legend_handles_labels()  # Get from last axis
                fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98),
                           ncol=3, fontsize=fontsize, framealpha=0.9)

                # Remove the individual legend from the first plot
                # (delete or comment out the legend creation inside the "if col_idx == 0:" block)


                ax.tick_params(axis='both', labelsize=fontsize - 2)

    return fig


# =============================================================================
# Main
# =============================================================================

def main():
    # File paths - organized by speed and trajectory
    base_path = r"C:\Users\ltjth\Documents\Research\UKF_Data"

    # With wind data
    files_wind = {
        0.3: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF0.3ms_FAN1.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF0.3ms_FAN1.csv"
        },
        0.5: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF0.5ms_FAN1.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF0.5ms_FAN_run21.csv"
        },
        1.0: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF1.0ms_4.5m_FAN1.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF1.0ms_FAN1.csv"
        },
        1.5: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF1.5ms_4.5m_FAN1.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF1.5ms_4.5m_FAN_run21.csv"
        },
        2.0: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF2.0ms_4.5m_FAN1.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF2.0ms_4.5m_FAN1.csv"
        }
    }

    # No wind data
    files_no_wind = {
        0.3: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF0.31.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF0.3ms1.csv"
        },
        0.5: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF0.53.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF0.5ms_lowerheight21.csv"
        },
        1.0: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF1.0ms1.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF1.0ms1.csv"
        },
        1.5: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF1.5ms1.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF1.5ms1.csv"
        },
        2.0: {
            'Line': f"{base_path}\\CF_ARR_Line_EKF2.0ms_4.5m2.csv",
            'Square': f"{base_path}\\CF_ARR_Square_EKF2.0ms_4.5m1.csv"
        }
    }

    speeds = [0.3, 0.5, 1.0, 1.5, 2.0]

    # Define custom xlim for no-wind Line trajectories
    xlim_line_no_wind = {
        0.3: 25,
        0.5: 16,
        1.0: 7,
        1.5: 5,
        2.0: 6
    }

    # Define custom xlim for no-wind Square trajectories
    xlim_square_no_wind = {
        0.3: 130,
        0.5: 100,
        1.0: 60,
        1.5: 50,
        2.0: 50
    }

    # Define custom xlim for with-wind Line trajectories
    xlim_line_wind = {
        0.3: 25,
        0.5: 15,
        1.0: 10,
        1.5: 7,
        2.0: 6
    }

    # Define custom xlim for with-wind Square trajectories
    xlim_square_wind = {
        0.3: 120,
        0.5: 90,
        1.0: 60,
        1.5: 60,
        2.0: 50
    }

    # UKF parameters
    Q_airflow = 0.005
    R_odom = 0.003
    R_magnetic = np.array([
        [0.72422324, 0.33065],
        [0.33065, 0.76509166]
    ])

    print("=" * 60)
    print("SIDE-BY-SIDE WIND COMPARISON")
    print("=" * 60)

    results_no_wind = {}
    results_wind = {}

    # Process no-wind data
    print("\n--- Processing NO WIND data ---")
    for speed in speeds:
        print(f"\nProcessing {speed} m/s...")
        results_no_wind[speed] = {}

        for traj in ['Line', 'Square']:
            filepath = files_no_wind[speed][traj]
            print(f"  Loading {traj}...")

            try:
                df = load_data(filepath)
                results = run_ukf(df, Q_airflow=Q_airflow,
                                  R_magnetic=R_magnetic, R_odom=R_odom)
                results_no_wind[speed][traj] = results

                wind_mean = np.mean(results['wind_mag'])
                wind_std = np.std(results['wind_mag'])
                print(f"    Wind: {wind_mean:.3f} ± {wind_std:.3f} m/s")
            except FileNotFoundError:
                print(f"    File not found: {filepath}")
            except Exception as e:
                print(f"    Error: {e}")

    # Process with-wind data
    print("\n--- Processing WITH WIND data ---")
    for speed in speeds:
        print(f"\nProcessing {speed} m/s...")
        results_wind[speed] = {}

        for traj in ['Line', 'Square']:
            filepath = files_wind[speed][traj]
            print(f"  Loading {traj}...")

            try:
                df = load_data(filepath)
                results = run_ukf(df, Q_airflow=Q_airflow,
                                  R_magnetic=R_magnetic, R_odom=R_odom)
                results_wind[speed][traj] = results

                wind_mean = np.mean(results['wind_mag'])
                wind_std = np.std(results['wind_mag'])
                print(f"    Wind: {wind_mean:.3f} ± {wind_std:.3f} m/s")
            except FileNotFoundError:
                print(f"    File not found: {filepath}")
            except Exception as e:
                print(f"    Error: {e}")

    # Generate plot
    print("\nGenerating figure...")
    fig = plot_side_by_side_comparison(
        results_no_wind, results_wind, speeds,
        xlim_line_no_wind, xlim_square_no_wind,
        xlim_line_wind, xlim_square_wind,
        fontsize=11
    )

    # Save
    # fig.savefig('wind_comparison_side_by_side.pdf', dpi=300, bbox_inches='tight')
    # fig.savefig('wind_comparison_side_by_side.png', dpi=300, bbox_inches='tight')
    # print("Saved: wind_comparison_side_by_side.pdf/png")

    plt.show()


if __name__ == "__main__":
    main()